# 16-amaliyot (M4) — NLP modelini API sifatida joylashtirish (SentimentAPI)

**Mavzu:** O'qitilgan sentiment modelni FastAPI veb-xizmatiga aylantiramiz va Docker'ga o'raymiz.
**Kuni:** M4 (chorshanba, 8-iyul) · **Bog'liq ma'ruza:** [L16 — MLOps](../lectures/d16_mlops.pdf) (keyin keladi — flipped).
**Quriladigan kapstone artefakti:** `capstone/app.py` (SentimentAPI).

> ⚠️ **Bu notebook to'liq ishlangan (fully worked)** — bo'sh joy yo'q. Kodni o'qing, ishga tushiring,
> tekshiring va o'zgartiring (PRIMM). Yakuniy mahsulotni Docker bilan ham ishga tushira olasiz.

## 1. Muhit tekshiruvi

In [ ]:
import os, sys
OFFLINE_FALLBACK = True
for cand in ["../../capstone", "/kaggle/working/capstone", "capstone"]:
    if os.path.isdir(cand):
        sys.path.insert(0, cand); break
from fastapi.testclient import TestClient
print("FastAPI TestClient tayyor.")

## 2. Yaxlit natija (avval manzil)

`capstone/app.py` dagi FastAPI ilovasini yuklab, `POST /predict` ga so'rov yuboramiz.

In [ ]:
from app import create_sentiment_api
api = create_sentiment_api()
client = TestClient(api)
r = client.post("/predict", json={"text": "mahsulot juda sifatli va arzon"})
print(r.status_code, r.json())

## 3. Tayyor kod bloki — `app.py` (PRIMM)

`capstone/app.py` (to'liq beriladi) quyidagilarni qiladi: (1) m13 modelni **bir marta** (global) yuklaydi;
(2) `POST /predict` `{"text": ...}` ni qabul qiladi; (3) `{"sentiment", "confidence"}` JSON qaytaradi.

```python
m13.USE_TRANSFORMERS = False          # offline: TF-IDF + LogReg
_clf = m13.FineTunedClassifier(); _clf.fit(_TRAIN_TEXTS, _TRAIN_LABELS)

@app.post("/predict")
def predict(body: PredictIn):
    sentiment = _clf.predict(body.text)
    confidence = max(_clf.predict_proba(body.text).values())
    return {"sentiment": sentiment, "confidence": round(float(confidence), 4)}
```

### Bashorat qiling
`POST /predict` ga `{"text": "juda yomon xizmat"}` yuborsangiz, `sentiment` qaysi bo'ladi?

### Tekshiring
Quyidagi katakni ishga tushiring va javobni solishtiring.

In [ ]:
neg = client.post("/predict", json={"text": "juda yomon xizmat, umuman yoqmadi"})
print(neg.json())
root = client.get("/")
print(root.json())

### O'zgartiring
`text` ni o'z sharhingiz bilan almashtiring va `confidence` qanday o'zgarishini kuzating.

## 4. Dockerfile (konteyner)

Ilovani har joyda bir xil ishga tushirish uchun Docker image quramiz (lokal build ixtiyoriy):

```dockerfile
FROM python:3.11-slim
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY capstone/ ./capstone/
EXPOSE 8000
CMD ["uvicorn", "capstone.app:app", "--host", "0.0.0.0", "--port", "8000"]
```
Lokal ishga tushirish: `docker build -t sentiment-api .` so'ng `docker run -p 8000:8000 sentiment-api`.

## 5. Loyihaga ulash va tekshirish

`capstone/app.py` tayyor. Quyidagi tekshiruv `POST /predict` shartnomasini tasdiqlaydi.

In [ ]:
resp = client.post("/predict", json={"text": "mahsulot juda sifatli va arzon"}).json()
print("Javob:", resp)
assert resp["sentiment"] in ("ijobiy", "salbiy")
assert 0.0 <= resp["confidence"] <= 1.0
print("To'g'ri! /predict -> {sentiment in (ijobiy, salbiy), confidence in [0,1]}.")

**Git:**
```bash
git add capstone/app.py
git commit -m "M4: SentimentAPI (FastAPI) deploy"
git push
```

## 6. Yakun — kapstone himoyasi

SentimentAPI tayyor. Kapstone himoyasida uni `m15 DocumentAssistantAgent` bilan birga ko'rsating:
agent o'zbekcha so'rovni qabul qiladi, kerakli vositani (sentiment/RAG/xulosa) chaqiradi; SentimentAPI
esa modelni HTTP orqali ochadi. **16 kunlik mehnatingiz — bitta ishlaydigan tizim. Tabriklaymiz!** 🎓

**Chiqish chiptasi:** Kapstone loyihangizni real hayotda qayerda qo'llagan bo'lardingiz? (Bir jumla.)